# Fill-in-the-blank: build eFlesh's forearm lattice

This worksheet reconstructs **`eFlesh/generate_lattice_v10_5.py`** -- the script
that produced the pad in the armband. Fill in the blanks and you export the same
`lattice_v10_5.stl`, from the same parameters, through the same steps.

Pressing the pad squeezes a soft lattice, the four embedded magnets move a
fraction of a millimetre, and the MLX90393 magnetometers underneath read that
motion as a change in field. A *stiff* pad reads nothing, so nearly every number
below is a decision about compliance -- and most were settled against real
prints, not simulation.

Fill in each marked `None` in order. Every exercise gives you the shape of the
answer as a code skeleton with `???` where your part goes.

| # | Step | v10_5 function | Minutes |
|---|------|----------------|---------|
| 1 | Input surface + grid shift | `build_input_surface` | 4 |
| 2 | Cell grid and per-layer stiffness | `young`, `run_cut_cells_cli` | 6 |
| 3 | Load the inflated lattice | `cut_cells_cli` output | 3 |
| 4 | Clean the raw mesh | `drop_tiny_debris` | 4 |
| 5 | Carve one magnet plug | `build_pocket_plug` | 6 |
| 6 | Weld the plugs in | `add_magnet_pockets` | 4 |
| 7 | Base plate | `build` phase 3 | 4 |
| 8 | Verify the pockets | `verify_pockets_*` | 4 |
| 9 | Export | `fix_stl_roundtrip`, `main` | 3 |

**About 38 minutes.** No 3D printer or hardware needed. The geometry itself
computes in about 35 seconds, most of it in the eight boolean operations of
Exercises 6 and 7.

> **One step does not run here.** The microstructure itself is produced by
> `cut_cells_cli`, a C++ program that only builds on Linux and takes about an
> hour to compile. It was run once by the repository's
> *Generate raw lattice (cut_cells_cli)* GitHub Action, and its output is
> committed to `notebooks/week2/data/`. Exercise 3 loads that file. Everything
> else is yours to rebuild, and runs identically on macOS and Windows.


## Setup

From the repository root --- PowerShell on Windows:

```powershell
.\.venv\Scripts\python.exe -m pip install -r .\notebooks\week2\requirements.txt
.\.venv\Scripts\python.exe -m jupyter lab
```

Terminal on macOS:

```bash
./.venv/bin/python -m pip install -r notebooks/week2/requirements.txt
./.venv/bin/python -m jupyter lab
```

Run the next cell. Everything you generate lands in `notebooks/week2/output/`,
created on first use and git-ignored.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import trimesh


# Finds the folder holding this notebook and week2_lattice.py.
def find_week2_dir(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "week2_lattice.py").exists():
            return candidate
        if (candidate / "notebooks" / "week2" / "week2_lattice.py").exists():
            return candidate / "notebooks" / "week2"
    raise FileNotFoundError("Open Jupyter from inside the project repository.")


sys.path.insert(0, str(find_week2_dir()))

from week2_lattice import (
    describe_mesh,
    drop_tiny_debris,
    find_repo_root,
    load_material2geometry,
    load_raw_lattice,
    output_dir,
    plot_side_section,
    slicer_notes,
)

REPO_ROOT = find_repo_root()
OUT = output_dir()
print("repository:", REPO_ROOT)
print("output to: ", OUT)


## Exercise 1 -- the input surface, and the grid shift (~4 min)

eFlesh's inflater fills whatever closed surface you hand it, so the box *is* the
sensor's outline: **30 x 30 mm** in plan, **13 mm** of lattice. (A 1 mm base
plate goes on in Exercise 7, making the finished part 14 mm tall.)

But the box does not sit at the origin. v10_5 shifts it by **`GRID_SHIFT = 1.0`
mm** in X and Y, and that shift is load-bearing.

The inflater's cell grid is anchored to the **world** origin, not to your part.
With the box at 0-30 its centre is at 15 mm -- the middle of a cell. The four
magnet pockets at 9 and 21 mm then land 1 mm past one grid line and 3 mm before
the next, so the magnets sit in *different* positions within the cell pattern and
get different amounts of lattice supporting them. Four sensors that should match
would not.

Shift the box by 1 mm and its centre lands at 16 mm, exactly on a cell boundary.
The pockets move to 10 and 22 mm, a mirror pair about the centre, and all four
become identical. Exercise 8 checks it; eFlesh's `verify_pockets_symmetric` is
the regression guard for a real v8.1 bug where one pocket came out unlike the
other three.

**Goal:** build the shifted box.

**Target:** `pad_box.bounds` is `[[1, 1, 0], [31, 31, 13]]`, 12 triangles.

**Hint:** `trimesh.creation.box` centres the box on the origin, so each entry
moves it into the positive octant *and then* adds the shift. Z is not shifted:

```python
to_grid = [BOX_X / 2 + ???, ???, LATTICE_H / 2]
```


In [ ]:
# TODO: replace each None with the dimension in millimetres.
BOX_X = None
BOX_Y = None
LATTICE_H = None
GRID_SHIFT = None     # X/Y offset that lands the pad centre on a cell boundary

assert None not in (BOX_X, BOX_Y, LATTICE_H, GRID_SHIFT), "Fill in all four."

pad_box = trimesh.creation.box(extents=[BOX_X, BOX_Y, LATTICE_H])

# TODO: replace None with the translation. See the hint.
to_grid = None

assert to_grid is not None, "A three-element list."
pad_box.apply_translation(to_grid)

box_path = OUT / "forearm_pad_box_v10_5.obj"
pad_box.export(box_path)

assert len(pad_box.faces) == 12, "A box is 12 triangles."
np.testing.assert_allclose(pad_box.bounds, [[1, 1, 0], [31, 31, 13]], atol=1e-9)
centre = (pad_box.bounds[0] + pad_box.bounds[1]) / 2
print(f"pad centre now at x={centre[0]:.1f}, y={centre[1]:.1f} mm "
      f"(both multiples of the 8 mm cell)")
print("Exercise 1 passed -- wrote", box_path.name)


## Exercise 2 -- the cell grid and per-layer stiffness (~6 min)

This is the genuinely unusual part of eFlesh. You do not pick strut thicknesses.
You pick a **target Young's modulus**, and a fitted spline map inverts it into
the 13 geometry parameters of cut-cell pattern `0646`. That map is
`material2geometry`, and it is pure Python -- it runs right here.

v10_5 asks for a different modulus per Z layer:

```python
YOUNG_BY_K = {-1: 0.0001, 0: 0.0001, 1: 0.0002}
```

The lower two cell layers are the softest the inversion will go; the top layer is
twice as stiff. That is deliberate. Only lattice *below* a magnet can compress
and carry it toward the sensor -- roughly half the pad's compliance sits above
the magnets and does nothing for the reading -- so the softness is spent low and
the touch surface is left firmer.

The cell indices come from dividing the **shifted** bounding box by the cell
size, exactly as `cut-cell.ipynb` does: `ceil(min) - 1` and `floor(max)`. The
extra cell on the low side starts the grid outside the part so no strut stops
short of a face.

**Goal:** fill in the lattice parameters and reproduce the index range.

**Target:** `corner0` is `[0, 0, -1]`, `corner1` is `[3, 3, 1]` -- 48 cells.

**Hint:** `lo` and `hi` are 3-element arrays; keep them arrays:

```python
corner0 = np.ceil(??? / CELL_SIZE).astype(int) - 1
corner1 = np.floor(??? / CELL_SIZE).astype(int)
YOUNG_BY_K = {-1: 0.0001, 0: ???, 1: ???}
```


In [ ]:
RESOLUTION = 50       # given: inflater samples per cell edge

# TODO: replace each None.
CELL_SIZE = None      # mm, edge length of one unit cell
NU = None             # target Poisson's ratio
YOUNG_BY_K = None     # dict: cell layer k -> target Young's modulus

assert None not in (CELL_SIZE, NU, YOUNG_BY_K), "Fill in all three."

lo, hi = pad_box.bounds

# TODO: replace both None values with the first and last cell index.
corner0 = None
corner1 = None

assert corner0 is not None and corner1 is not None, "Divide the bounds by CELL_SIZE."
n_cells = int(np.prod(corner1 - corner0 + 1))

assert list(corner0) == [0, 0, -1], f"corner0 should be [0, 0, -1], got {list(corner0)}"
assert list(corner1) == [3, 3, 1], f"corner1 should be [3, 3, 1], got {list(corner1)}"
assert n_cells == 48, f"Expected 48 cells, got {n_cells}"
assert YOUNG_BY_K[0] == 0.0001 and YOUNG_BY_K[1] == 0.0002, "Check the two moduli."
print(f"{n_cells} cells: {(corner1 - corner0 + 1)} across x, y, z")


Now run the inversion itself -- the same call `generate_lattice_v10_5.py` makes
once per cell. `evaluate(nu, E)` returns 13 parameters; entries 5 to 8 are strut
thicknesses as a fraction of a cell, so multiplying by `CELL_SIZE` gives
millimetres. That is how you check a lattice is printable before generating it.


In [ ]:
mat2geo = load_material2geometry(REPO_ROOT)

for k, E in sorted(YOUNG_BY_K.items()):
    thicknesses = [p * CELL_SIZE for p in mat2geo.evaluate(NU, E)[5:9]]
    verdict = "printable" if min(thicknesses) >= 0.4 else "TOO THIN for a 0.4 mm nozzle"
    print(f"layer k={k:+d}  E={E:<8} -> thinnest strut {min(thicknesses):.3f} mm   {verdict}")

thinnest = min(p * CELL_SIZE for p in mat2geo.evaluate(NU, YOUNG_BY_K[0])[5:9])
assert abs(thinnest - 0.628) < 0.002, f"Expected 0.628 mm, got {thinnest:.3f}"
print("\n0.628 mm -- the figure quoted in the v10_5 docstring.")


## Exercise 3 -- load the inflated lattice (~3 min)

With those parameters written to `data.json`, v10_5 shells out to the inflater:

```
cut_cells_cli -p data.json --surface shapes/forearm_pad_box_v10_5.obj \
              --gridSize 8.0 -o output/forearm_lattice_raw_v10_5.obj -r 50
```

That binary is the one thing your laptop cannot run, so its output is committed
to `notebooks/week2/data/`. Load it and check it against the box you built --
if the two disagree, the lattice was inflated for different parameters and
nothing downstream would be reproducing v10_5.

**Goal:** load the lattice and assert it fills your shifted box.

**Target:** a mesh of several hundred thousand triangles spanning
`[[1, 1, 0], [31, 31, 13]]`.

**Hint:** you already have an object whose `.bounds` is exactly what the
lattice's should be.

```python
expected_bounds = ???.bounds
```


In [ ]:
raw = load_raw_lattice()
describe_mesh(raw, "raw lattice from cut_cells_cli")

# TODO: replace None -- what should the lattice's bounds equal?
expected_bounds = None

assert expected_bounds is not None, "You built this in Exercise 1."
np.testing.assert_allclose(raw.bounds, expected_bounds, atol=0.05)
print("\nLattice matches the input surface -- same parameters, same part.")


## Exercise 4 -- clean the raw mesh (~4 min)

Marching an isosurface over a cell grid leaves artefacts: degenerate triangles,
duplicated vertices at cell seams, and small disconnected shells where the
lattice meets the box faces. v10_5 clears them before any boolean, because a
boolean on a dirty mesh fails in ways that are hard to see.

The last step, `drop_tiny_debris`, carries a scar. In **v8.1** it deleted every
magnet cavity: a cavity's boundary is *topologically its own small closed shell*,
so the debris filter ate all four. The mesh stayed watertight, stayed the right
size, sliced fine -- and printed with nowhere to put the magnets. The fix was a
`protect_xy` argument exempting shells near the pocket centres.

There are no pockets yet, so nothing needs protecting. In Exercise 6, after the
plugs go in, it is called again *with* `protect_xy` -- watch for it.

**Goal:** run v10_5's cleanup sequence.

**Target:** still watertight, same bounds.

**Hint:** all three take no arguments and act in place. Give their names as
strings:

```python
for method_name in ("merge_vertices", ???, ???):
```

in this order: collapse duplicated vertices at the cell seams, drop vertices no
face references any more, then make every normal point outward.


In [ ]:
before = len(raw.faces)
raw.update_faces(raw.nondegenerate_faces())

# TODO: replace each None with a method name, as a string. See the hint.
for method_name in (None, None, None):
    assert method_name is not None, "Three cleanup methods, in order."
    getattr(raw, method_name)()

raw = drop_tiny_debris(raw)

describe_mesh(raw, "cleaned lattice")
assert raw.is_watertight, "Cleanup should leave a closed surface."
np.testing.assert_allclose(raw.bounds, pad_box.bounds, atol=0.05)
print(f"\n{before:,} -> {len(raw.faces):,} triangles after cleanup")


## Exercise 5 -- carve one magnet plug (~6 min)

Four N52 magnets sit inside the pad, **4.8 mm diameter x 1.6 mm thick**, one over
each magnetometer, seating at **z = 6.5 mm**.

They cannot simply be holes. Measure the lattice inside one magnet-sized cylinder
and you find under 3 mm^3 of solid in a 29 mm^3 volume -- subtracting a cylinder
from 90% air removes almost nothing and leaves no pocket. So v10_5 drops in a
**solid plug** and carves the cavity into *that*. Its four numbers each have a
history:

* **`CAVITY_CLEARANCE = 0.25 mm`** on the radius (0.5 on the diameter) so the
  magnet actually drops in.
* **`PLUG_FLOOR_MARGIN = 0.8 mm`** of solid under the magnet. Thinner is more
  sensitive -- compliant lattice under the magnet is the only material whose
  compression moves the magnet toward the sensor. But that floor has to bridge
  over lattice while printing: 0.4 mm sagged, 0.6 mm still seated the magnet
  roughly, 0.8 mm (4 layers at 0.2) finally printed flat. Do not go below 0.6.
* **`PLUG_ROOF_MARGIN = 1.0 mm`** of solid above, printed after the pause, which
  is what retains the magnet.
* **`POCKET_BOTTOM_OVERCUT = 0.10 mm`** so the cavity is a hair deeper than the
  magnet and never leaves it standing proud.

**Goal:** fill in the cavity radius, the plug's Z extent, and the carve.

**Target:** plug spans z 4.0 to 7.5, cavity z 4.8 to 6.5, watertight.

**Hint:** the plug is the cavity pushed out by the floor below and the roof
above; a cylinder's centre is the average of its two ends:

```python
CAVITY_RADIUS = MAGNET_RADIUS + ???
PLUG_Z_MIN = CAVITY_Z_MIN - ???
PLUG_Z_MAX = CAVITY_Z_MAX + ???
```


In [ ]:
# Given: v10_5's magnet and plug constants.
MAGNET_RADIUS = 4.8 / 2.0
MAGNET_DEPTH = 1.6
PAUSE_Z_LOC = 6.5
POCKET_BOTTOM_OVERCUT = 0.10
CAVITY_CLEARANCE = 0.25
PLUG_FLOOR_MARGIN = 0.8
PLUG_ROOF_MARGIN = 1.0
PLUG_RADIUS = MAGNET_RADIUS + 1.0

CAVITY_DEPTH = MAGNET_DEPTH + POCKET_BOTTOM_OVERCUT
CAVITY_Z_MAX = PAUSE_Z_LOC
CAVITY_Z_MIN = CAVITY_Z_MAX - CAVITY_DEPTH

# TODO: replace each None.
CAVITY_RADIUS = None
PLUG_Z_MIN = None
PLUG_Z_MAX = None

assert None not in (CAVITY_RADIUS, PLUG_Z_MIN, PLUG_Z_MAX), "Fill in all three."
PLUG_HEIGHT = PLUG_Z_MAX - PLUG_Z_MIN

assert abs(CAVITY_RADIUS - 2.65) < 1e-9, f"cavity radius should be 2.65, got {CAVITY_RADIUS}"
assert abs(PLUG_Z_MIN - 4.0) < 1e-9, f"plug should start at z = 4.0, got {PLUG_Z_MIN}"
assert abs(PLUG_Z_MAX - 7.5) < 1e-9, f"plug should end at z = 7.5, got {PLUG_Z_MAX}"
print(f"plug   z {PLUG_Z_MIN} .. {PLUG_Z_MAX} mm, radius {PLUG_RADIUS} mm")
print(f"cavity z {CAVITY_Z_MIN} .. {CAVITY_Z_MAX} mm, radius {CAVITY_RADIUS} mm")


In [ ]:
def build_pocket_plug():
    """v10_5's build_pocket_plug: a solid plug with one blind cavity."""
    plug = trimesh.creation.cylinder(radius=PLUG_RADIUS, height=PLUG_HEIGHT, sections=64)
    cavity = trimesh.creation.cylinder(radius=CAVITY_RADIUS, height=CAVITY_DEPTH + 0.02, sections=64)

    # TODO: replace both None values with each cylinder's mid-height.
    plug_centre_z = None
    cavity_centre_z = None

    assert None not in (plug_centre_z, cavity_centre_z), "Average each span's two ends."
    plug.apply_translation([0.0, 0.0, plug_centre_z])
    cavity.apply_translation([0.0, 0.0, cavity_centre_z])

    # TODO: replace None -- carve the cavity out of the plug.
    carved = None

    assert carved is not None, 'Use .difference(..., engine="manifold").'
    return carved


one_plug = build_pocket_plug()
np.testing.assert_allclose(one_plug.bounds[:, 2], [PLUG_Z_MIN, PLUG_Z_MAX], atol=1e-6)
assert one_plug.is_watertight, "The carve left holes."
assert one_plug.volume < np.pi * PLUG_RADIUS ** 2 * PLUG_HEIGHT - 10, "Nothing got carved out."
print(f"one plug: {len(one_plug.faces)} triangles, {one_plug.volume:.1f} mm^3 of solid")


## Exercise 6 -- weld the plugs into the lattice (~4 min)

Two booleans per pocket, four pockets. The slowest cell in the notebook, about
twelve seconds.

1. **Clear a socket** -- subtract a cylinder so no struts are left where the plug
   is going. Union-ing a plug into a tangle of struts makes a mesh full of
   near-degenerate slivers.
2. **Drop the plug in** -- union it.

Two details, both load-bearing:

* The socket is cut **narrower** than the plug, by `CLEARANCE_OVERLAP = 0.15 mm`,
  so the plug interpenetrates the surrounding lattice and welds to it. Cut it
  plug-sized and the two solids merely touch along a coincident surface, which
  booleans and slicers both handle badly.
* The socket's extra 0.5 mm of height goes entirely **above** the plug. v10_4
  centred it instead, which cut away the lattice *under* the floor -- leaving the
  0.8 mm floor bridging over open void, where it sagged and strung into the
  cavity. The lattice underneath has to survive to hold the floor up.

The pocket centres are given in part coordinates, so they need `GRID_SHIFT` added
to reach the shifted lattice.

**Goal:** place all four plugs.

**Hint:**

```python
x, y = x0 + ???, y0 + ???
socket_radius = PLUG_RADIUS - ???
lattice = lattice.union(???, engine="manifold")
```


In [ ]:
# Given: v10_5's pocket layout, from eFlesh/magnet_layout.py.
POCKET_CENTERS = ((9.0, 9.0), (21.0, 9.0), (9.0, 21.0), (21.0, 21.0))
POLARITY = "N S N S"
CLEARANCE_OVERLAP = 0.15

lattice = raw.copy()
for idx, (x0, y0) in enumerate(POCKET_CENTERS, start=1):
    # TODO: replace both None values -- the centre in the shifted lattice.
    x, y = None, None

    assert None not in (x, y), "The lattice is shifted; the centres are not."
    clr_h = PLUG_HEIGHT + 0.5

    # TODO: replace None with the socket radius.
    socket_radius = None

    assert socket_radius is not None, "Narrower than PLUG_RADIUS; see the hint."
    clearance = trimesh.creation.cylinder(radius=socket_radius, height=clr_h, sections=64)
    clearance.apply_translation([x, y, PLUG_Z_MIN + clr_h / 2.0])
    lattice = lattice.difference(clearance, engine="manifold")

    plug = build_pocket_plug()
    plug.apply_translation([x, y, 0.0])

    # TODO: replace None -- weld this plug into the lattice.
    lattice = None

    assert lattice is not None, "Union the plug in."
    print(f"  pocket {idx}: part ({x0}, {y0}) -> lattice ({x}, {y}), "
          f"{len(lattice.faces):,} faces")

# The debris pass that ate every pocket in v8.1 -- now told what to protect.
shifted_centers = tuple((x + GRID_SHIFT, y + GRID_SHIFT) for x, y in POCKET_CENTERS)
lattice = drop_tiny_debris(lattice, protect_xy=shifted_centers)

describe_mesh(lattice, "lattice + plugs")
assert lattice.is_watertight, "A boolean left holes."


## Exercise 7 -- shift back and add the base plate (~4 min)

The lattice has done its job on the shifted grid, so it moves back to the origin
-- and up by `BASE_H`, to sit on the 1 mm solid plate the sensor board mounts
against.

Then the plate is unioned on. Note what v10_5 does first:

```python
for _ in range(3):
    base = base.subdivide()
```

A box is 12 triangles. Union-ing 12 triangles against a lattice of several
hundred thousand is exactly where **v10.1 lost its base plate entirely** -- the
output came out with no plate and nothing reported an error. Subdividing three
times (12 -> 768 triangles) gives the boolean enough vertices on the plate to
behave. `BASE_OVERLAP` makes the plate 0.02 mm taller than 1 mm so it bites into
the lattice rather than merely touching it.

**Goal:** shift back, build the plate, union.

**Target:** watertight, spanning `[[0, 0, 0], [30, 30, 14]]`.

**Hint:** the shift undoes X and Y and raises Z; the plate's centre is half its
own height off the bed:

```python
back_to_origin = [-GRID_SHIFT, ???, ???]
base_centre = [BOX_X / 2, BOX_Y / 2, ???]
final = base.union(???, engine="manifold")
```


In [ ]:
BASE_H = 1.0
BASE_OVERLAP = 0.02

stacked = lattice.copy()

# TODO: replace both None values.
back_to_origin = None   # undo GRID_SHIFT in x and y, and lift by the base height
base_centre = None      # centre of the plate, whose bottom sits at z = 0

assert None not in (back_to_origin, base_centre), "Fill in both."
stacked.apply_translation(back_to_origin)

base = trimesh.creation.box(extents=(BOX_X, BOX_Y, BASE_H + BASE_OVERLAP))
base.apply_translation(base_centre)
for _ in range(3):
    base = base.subdivide()

# TODO: replace None -- union the lattice onto the plate.
final = None

assert final is not None, "Union them."
final = drop_tiny_debris(final, protect_xy=POCKET_CENTERS)
final.fix_normals()

describe_mesh(final, "final part")
assert final.is_watertight, "The base union left holes."
np.testing.assert_allclose(final.bounds, [[0, 0, 0], [30, 30, 14]], atol=0.05)

plot_side_section(final.triangles, y=9.0,
                  title="Section at y = 9 mm -- two plugs, cavities, 1 mm base")


## Exercise 8 -- verify the pockets (~4 min)

v10_5 ships two regression guards, and both exist because of real failures.

**`verify_pockets_open`** -- is the middle of each cavity actually empty? In v8.1
the debris pass sealed all four solid and every other check still passed.
`mesh.contains` answers it directly.

**`verify_pockets_symmetric`** -- do all four pockets look alike? Count triangles
inside the clearance radius over the plug's Z range. The only geometry there
should be the cavity wall, so the count must be *identical* at every pocket. In
v8.1 the lattice failed to clear fully at (21, 21) and that pocket differed. It
is also the check that catches a missing `GRID_SHIFT`.

**Goal:** fill in the two tests.

**Target:** nothing solid at any cavity midpoint, one identical face count at all
four pockets.

**Hint:** everything is now in part coordinates, lifted by `BASE_H`:

```python
mid_z = (CAVITY_Z_MIN + CAVITY_Z_MAX) / 2 + ???
is_solid = final.contains(???)
dist = np.sqrt((centres[:, 0] - x) ** 2 + (centres[:, 1] - y) ** 2)
```


In [ ]:
# --- verify_pockets_open ---------------------------------------------
# TODO: replace None -- height of the cavity midpoint in the FINAL part.
mid_z = None

assert mid_z is not None, "The part was lifted by the base plate."
test_points = np.array([[x, y, mid_z] for x, y in POCKET_CENTERS])

# TODO: replace None -- is each test point inside solid material?
is_solid = None

assert is_solid is not None, "Use final.contains(test_points)."
for (x, y), solid in zip(POCKET_CENTERS, is_solid):
    assert not solid, f"Pocket at ({x}, {y}) is SOLID at its midpoint -- it got sealed."
print(f"All 4 cavities open at z = {mid_z:.2f} mm")

# --- verify_pockets_symmetric ----------------------------------------
centres = final.triangles_center
zlo = PLUG_Z_MIN + 0.05 + BASE_H
zhi = PLUG_Z_MAX - 0.05 + BASE_H
clear_r = PLUG_RADIUS - CLEARANCE_OVERLAP - 0.05

counts = []
for x, y in POCKET_CENTERS:
    # TODO: replace None -- horizontal distance from each face centre to (x, y).
    dist = None

    assert dist is not None, "Use the x and y columns of centres."
    mask = (dist < clear_r) & (centres[:, 2] > zlo) & (centres[:, 2] < zhi)
    counts.append(int(np.count_nonzero(mask)))

for (x, y), count, pole in zip(POCKET_CENTERS, counts, POLARITY.split()):
    print(f"  pocket ({x:4.1f}, {y:4.1f})  pole {pole}  {count:5d} faces inside the clearance")

assert len(set(counts)) == 1, f"Pockets are NOT identical: {counts} -- check GRID_SHIFT."
print(f"\nAll four pockets identical at {counts[0]} faces. GRID_SHIFT did its job.")


## Exercise 9 -- export (~3 min)

One wrinkle at the end. STL has no shared-vertex topology, so reloading an STL
re-infers connectivity from raw float32 coordinates, and that can leave hairline
non-manifold seams at boolean boundaries. v10_5 exports, reloads, and runs a
`fix_stl_roundtrip` pass -- deliberately just a vertex merge, with *no*
repair-and-patch cycle, because in v8.1 that cascade deleted a pocket's cavity
floor and walls.

**Goal:** export, reload, merge, export again.

**Target:** `lattice_v10_5.stl` in `notebooks/week2/output/`, watertight,
30 x 30 x 14 mm.

**Hint:** name the file after the version you just rebuilt. `digits_vertex=5` is
the tolerance v10_5 merges at.


In [ ]:
# TODO: replace None with the output filename.
FINAL_NAME = None

assert FINAL_NAME is not None, "Name it after the version you just rebuilt."
final_path = OUT / FINAL_NAME
final.export(final_path)

reloaded = trimesh.load(final_path, force="mesh")
reloaded.update_faces(reloaded.nondegenerate_faces())

# TODO: replace None with the merge tolerance.
merge_digits = None

assert merge_digits is not None, "See the hint."
reloaded.merge_vertices(digits_vertex=merge_digits)
reloaded.update_faces(reloaded.nondegenerate_faces())
reloaded.remove_unreferenced_vertices()
reloaded.fix_normals()
reloaded = drop_tiny_debris(reloaded, protect_xy=POCKET_CENTERS)
reloaded.export(final_path)

describe_mesh(reloaded, "on disk")
assert final_path.name == "lattice_v10_5.stl"
np.testing.assert_allclose(reloaded.bounding_box.extents, [30, 30, 14], atol=0.05)
print(f"\nwrote {final_path}  ({final_path.stat().st_size / 1e6:.1f} MB)")
print("This is the part. Ready to slice.")


## Your print settings

The key line is the pause layer. The nozzle reaches the top of the four open
cavities, you pause, seat the magnets N-S-N-S in the printed order, and resume --
the next millimetre is the roof that seals them in.

Alternating poles matter: if every magnet pointed the same way their fields would
add into one broad blob and no sensor could tell which magnet moved.


In [ ]:
print(slicer_notes(PAUSE_Z_LOC, BASE_H, POLARITY, POCKET_CENTERS))


## Compare against the real thing

`eFlesh/output/lattice_v10_5.stl` is the file that was actually printed. Yours
came from the same raw lattice through the same steps.


In [ ]:
reference = trimesh.load(REPO_ROOT / "eFlesh/output/lattice_v10_5.stl", force="mesh")
rows = [
    ("triangles", f"{len(reloaded.faces):,}", f"{len(reference.faces):,}"),
    ("solid volume (mm^3)", f"{reloaded.volume:,.1f}", f"{reference.volume:,.1f}"),
    ("bbox x (mm)", f"{reloaded.bounding_box.extents[0]:.2f}", f"{reference.bounding_box.extents[0]:.2f}"),
    ("bbox y (mm)", f"{reloaded.bounding_box.extents[1]:.2f}", f"{reference.bounding_box.extents[1]:.2f}"),
    ("bbox z (mm)", f"{reloaded.bounding_box.extents[2]:.2f}", f"{reference.bounding_box.extents[2]:.2f}"),
]
print(f"{'':22}{'yours':>14}{'eFlesh v10_5':>16}")
for label, a, b in rows:
    print(f"{label:22}{a:>14}{b:>16}")


## What you can change, and what it costs

Every parameter above is live *except* the microstructure itself:

**Free to change here** -- re-run from Exercise 5, about 20 seconds:

* `PLUG_FLOOR_MARGIN` -- the sensitivity knob with teeth. Thinner means more
  compliant lattice under the magnet, the only material whose compression moves
  the magnet toward the sensor. eFlesh shipped 0.4, then 0.6, then 0.8 mm; the
  first two sagged while bridging. Do not go below 0.6 without a test print.
* `PAUSE_Z_LOC` -- lower it and signal climbs steeply (field falls off as
  1/r^4), but the lattice under each magnet gets shorter and has less travel.
* `POCKET_CENTERS`, `BASE_H`, `CAVITY_CLEARANCE`, `PLUG_ROOF_MARGIN`,
  `CLEARANCE_OVERLAP`.

**Needs the inflater re-run on Linux** -- the GitHub Action, roughly an hour:

* `CELL_SIZE`, `NU`, `YOUNG_BY_K`, `RESOLUTION`, `BOX_X`, `BOX_Y`, `LATTICE_H`,
  `GRID_SHIFT`.

Change one of those and the committed raw lattice no longer matches your
parameters -- which is exactly what Exercise 3's bounds check is there to catch.
`notebooks/week2/data/README.md` has the procedure.
